# CP-SAT Validation & Benchmark (Kaggle / Colab)

OR-Tools' CP-SAT solver hangs on macOS 15.x with ortools 9.15.x, so this notebook runs the validation on Linux (Kaggle / Colab) instead.

**What this notebook does:**
1. Install OR-Tools + repo dependencies
2. Clone the repo (or use a mounted path)
3. Sanity-check the CP-SAT solver on a trivial 1-var problem
4. Direct CP-SAT smoke test on a 5-pallet 40HC voyage
5. Run the full pytest suite for `test_cpsat.py`
6. Run `scripts/benchmark_cpsat.py` — CP-SAT vs 5 heuristics vs GA
7. Print the CSV summary

## 1. Install dependencies

In [ ]:
!pip install -q 'ortools>=9.10,<10' \
  'pydantic==2.9.2' 'pydantic-settings==2.6.1' \
  fastapi 'uvicorn[standard]' python-multipart websockets orjson loguru \
  numpy gymnasium deap shapely pandas \
  'pytest>=8' pytest-asyncio

## 2. Locate / clone the repo

Pinned to branch `cpsat-baseline` (change `BRANCH` to `'main'` once merged).

The cell below tries common locations (Kaggle dataset, `/kaggle/working`, `/content`, `./`) and looks for `app/algorithms/cpsat.py` as a sentinel. If no clone has it, a fresh shallow clone of the branch happens. Safe to re-run after a Colab reconnect — stale clones missing the sentinel are wiped first.

For a **private** repo, set `GITHUB_TOKEN` first (Kaggle: *Add-ons → Secrets*; Colab: `from google.colab import userdata; os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')`).

In [ ]:
import os, subprocess, sys, pathlib, shutil

REPO_URL = 'https://github.com/Seif-Sameh/loading-service-2.git'
BRANCH = 'cpsat-baseline'   # change to 'main' once this branch is merged
SENTINEL = 'app/algorithms/cpsat.py'  # file proving the clone has the CP-SAT code

candidates = [
    '/kaggle/working/loading-service-2',
    '/content/loading-service-2',
    './loading-service-2',
]
REPO_DIR = next((p for p in candidates if pathlib.Path(p, SENTINEL).exists()), None)

if REPO_DIR is None:
    # Either nothing cloned, or a stale clone on a branch missing cpsat.py.
    # Pick a writable target and (re)clone.
    target = '/kaggle/working/loading-service-2' if pathlib.Path('/kaggle/working').exists() else '/content/loading-service-2'
    if pathlib.Path(target).exists():
        shutil.rmtree(target)
    # Also check a read-only Kaggle dataset mount
    ro = pathlib.Path('/kaggle/input/loading-service-2')
    if ro.exists() and (ro / SENTINEL).exists():
        shutil.copytree(ro, target)
    else:
        tok = os.environ.get('GITHUB_TOKEN', '').strip()
        url = REPO_URL.replace('https://', f'https://{tok}@') if tok else REPO_URL
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, target])
    REPO_DIR = target

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('REPO_DIR =', REPO_DIR)
print('has', SENTINEL, ':', pathlib.Path(REPO_DIR, SENTINEL).exists())
print('contents:', sorted(os.listdir(REPO_DIR))[:20])

## 3. OR-Tools sanity — does CP-SAT even solve `x >= 5`?

Locally this hangs forever on macOS. On Linux this should finish in <0.1s.

In [ ]:
import time
from ortools.sat.python import cp_model
m = cp_model.CpModel()
x = m.NewIntVar(0, 10, 'x')
m.Add(x >= 5)
m.Maximize(x)
s = cp_model.CpSolver()
s.parameters.max_time_in_seconds = 5
t0 = time.perf_counter()
status = s.Solve(m)
print(f'status={s.StatusName(status)}  x={s.Value(x)}  in {time.perf_counter()-t0:.3f}s')
assert status == cp_model.OPTIMAL and s.Value(x) == 10, 'OR-Tools install broken'

## 4. Direct CP-SAT smoke test — 5 pallets in a 40HC

Calls `_solve_cpsat` directly with synthetic items. Bypasses the env/select replay so we can be certain the solver itself works before running the full benchmark.

In [ ]:
import os, sys, pathlib, time
# Self-contained: re-establish REPO_DIR after a possible kernel restart.
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

from app.algorithms.cpsat import CPSATConfig, _solve_cpsat
from app.catalog.loader import get_container, get_cargo_preset

container = get_container('40HC')
items = [get_cargo_preset('eur_pallet_light', item_id=f'p{i}') for i in range(5)]
cfg = CPSATConfig(time_limit_s=20.0, num_search_workers=2, grid_mm=100, log_search_progress=True)
t0 = time.perf_counter()
placements, status, obj = _solve_cpsat(container, items, cfg)
print(f'\nstatus={status}  planned={len(placements)}/{len(items)}  obj={obj}  time={time.perf_counter()-t0:.2f}s')
for p in placements:
    print(f'  {p.item_id}: pos=({p.position.x_mm},{p.position.y_mm},{p.position.z_mm})  rot={p.rotation}')

## 5. Unit tests

In [ ]:
import os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
!cd {REPO_DIR} && python -m pytest tests/test_cpsat.py -v --tb=short 2>&1 | tail -40

## 6. Full benchmark — CP-SAT vs heuristics vs GA

Tweak `--voyages`, `--items`, `--cpsat-time` for budget. Defaults: 5 voyages × 30 items, 30s CP-SAT budget per voyage (≈2.5 min total for CP-SAT alone).

In [ ]:
import os, sys, pathlib, time, csv, statistics
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

from app.algorithms import get_algorithm
from app.algorithms.base import solve
from app.catalog.loader import get_container
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig

# --- knobs ---
VOYAGES, ITEMS, CONTAINER, SEED = 5, 30, '40HC', 42
CPSAT_TIME, CPSAT_WORKERS = 30.0, 4
MORL_CKPT = ''   # set to a .pt path to include MORL-PCT
# -------------

sampler = AlexandriaSampler(SamplerConfig(n_items=ITEMS, strategy='mixed', seed=SEED))
cont = get_container(CONTAINER)
voyages = [(cont, sampler.sample()) for _ in range(VOYAGES)]

algo_codes = [
    ('bl', {}), ('extreme_points', {}), ('baf', {}), ('bssf', {}), ('blsf', {}),
    ('ga', {}),
    ('cpsat', {'time_limit_s': CPSAT_TIME, 'num_search_workers': CPSAT_WORKERS, 'enforce_imdg': True}),
]
if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
    algo_codes.append(('morl_pct', {'weights_path': MORL_CKPT, 'preference': [0.7, 0.1, 0.1, 0.05, 0.05]}))


def _ae(placements, container):
    if not placements:
        return 0.0
    door = 0.20 * container.internal.length_mm
    return sum(
        1 for p in placements
        if (p.position.x_mm + p.rotated_dimensions.length_mm / 2) <= door
    ) / len(placements)


rows = []
print(f'Building voyage suite: {VOYAGES} × {ITEMS} items, container={CONTAINER}')
for vi, (c, items) in enumerate(voyages):
    print(f'\n=== voyage {vi+1}/{VOYAGES}  ({len(items)} items) ===')
    for code, kwargs in algo_codes:
        algo = get_algorithm(code, **kwargs)
        t0 = time.perf_counter()
        if hasattr(algo, 'prepare'):
            algo.prepare(c, items)
        res, _ = solve(algorithm=algo, container=c, items=items)
        elapsed = time.perf_counter() - t0
        k = res.kpis
        ae = _ae(res.placements, c)
        ss = max(0.0, (len(res.placements) - k.unstable_count) / max(len(res.placements), 1))
        cps_status = str(algo.meta.get('cpsat_status', '')) if code == 'cpsat' else ''
        rows.append({
            'voyage': vi,
            'algorithm': code,
            'util_pct': 100 * k.utilization,
            'placed_pct': 100 * len(res.placements) / max(len(items), 1),
            'access_eff': ae,
            'stability_score': ss,
            'cog_long_abs': abs(k.cog_long_dev),
            'weight_pct': 100 * k.weight_used,
            'elapsed_s': elapsed,
            'cpsat_status': cps_status,
        })
        tag = f' [{cps_status}]' if cps_status else ''
        r = rows[-1]
        print(
            f"  {code:<16} util {r['util_pct']:>6.2f}%  placed {r['placed_pct']:>6.2f}%  "
            f"AE {ae:>5.2f}  SS {ss:>5.2f}  t {elapsed:>6.2f}s{tag}"
        )

# Aggregate
print('\n\n=== AGGREGATE (mean across voyages) ===')
print(
    f'{"algorithm":<16} {"util%":>7} {"std":>5} {"placed%":>8} {"AE":>5} {"SS":>5} '
    f'{"|CoG|":>6} {"wt%":>5} {"s":>6}'
)
print('-' * 75)
by = {}
for r in rows:
    by.setdefault(r['algorithm'], []).append(r)
for code, _ in algo_codes:
    rs = by.get(code, [])
    if not rs:
        continue
    utils = [r['util_pct'] for r in rs]
    u_std = statistics.stdev(utils) if len(utils) > 1 else 0.0
    print(
        f"{code:<16} {statistics.fmean(utils):>7.2f} {u_std:>5.2f} "
        f"{statistics.fmean(r['placed_pct'] for r in rs):>8.2f} "
        f"{statistics.fmean(r['access_eff'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['stability_score'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['cog_long_abs'] for r in rs):>6.3f} "
        f"{statistics.fmean(r['weight_pct'] for r in rs):>5.2f} "
        f"{statistics.fmean(r['elapsed_s'] for r in rs):>6.2f}"
    )

# CSV — each statement on its own line so paste-mangling can't break it.
out_dir = pathlib.Path(REPO_DIR, 'benchmarks/out')
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"cpsat_benchmark_{time.strftime('%Y%m%d_%H%M%S')}.csv"
with csv_path.open('w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print(f'\n→ wrote {csv_path}')

## 7. Show the CSV

In [ ]:
import pandas as pd, glob, os, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand; break
csvs = sorted(glob.glob(os.path.join(REPO_DIR, 'benchmarks/out/cpsat_benchmark_*.csv')))
if not csvs:
    print('No CSV found — did the benchmark run?')
else:
    df = pd.read_csv(csvs[-1])
    print('latest:', csvs[-1])
    summary = df.groupby('algorithm').agg(
        util_mean=('util_pct', 'mean'),
        util_std=('util_pct', 'std'),
        placed_mean=('placed_pct', 'mean'),
        access_eff=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
        time_s=('elapsed_s', 'mean'),
    ).round(2).sort_values('util_mean', ascending=False)
    print(summary)

---

## 8. Full evaluation suite (edge cases + PCT + MORL-PCT)

Stress-tests every algorithm across **6 voyage profiles** × multiple seeds:

| suite          | items | container | content                                  | what it stresses                |
|----------------|------:|-----------|------------------------------------------|---------------------------------|
| `small_easy`   |    12 | 40HC      | EUR pallets (preset)                     | CP-SAT should hit OPTIMAL fast  |
| `medium_mixed` |    30 | 40HC      | Wadaboa+presets mixed                    | baseline (matches §6)           |
| `large_mixed`  |    60 | 40HC      | Wadaboa+presets mixed                    | heuristics start to lose more   |
| `20gp_tight`   |    20 | 20GP      | mixed                                    | smaller envelope → density matters |
| `hazmat_heavy` |    24 | 40HC      | mix of class-3 drums, class-8 corrosives, plus filler | IMDG segregation        |
| `reefer`       |    12 | 20RF      | reefer fruit pallets only                | reefer container path           |

Upload your checkpoints to Colab (`/content/...`) and set the paths in the **next cell**. Leave a path empty (`''`) to skip that algorithm.

In [ ]:
# ========== CONFIG — edit these ==========
PCT_CKPT  = '/content/pct_latest.pt'              # '' to skip PCT
MORL_CKPT = '/content/morl_pct_latest.pt'         # '' to skip MORL-PCT
MORL_PREFERENCE = [0.7, 0.1, 0.1, 0.05, 0.05]     # util / access / stability / cog / weight

SEEDS = [42, 7, 123]                              # one full sweep per seed
CPSAT_TIME = 30.0                                 # seconds per voyage for CP-SAT
CPSAT_WORKERS = 4
RUN_GA = True                                     # GA is ~200s/voyage — set False to skip
SUITES_TO_RUN = [
    'small_easy', 'medium_mixed', 'large_mixed',
    '20gp_tight', 'hazmat_heavy', 'reefer',
]
# =========================================

import os, sys, pathlib
for _cand in ('/kaggle/working/loading-service-2', '/content/loading-service-2', './loading-service-2'):
    if pathlib.Path(_cand, 'app').exists():
        REPO_DIR = _cand
        os.chdir(REPO_DIR)
        if REPO_DIR not in sys.path:
            sys.path.insert(0, REPO_DIR)
        break

# Install torch only if a neural checkpoint is requested
_need_torch = bool(PCT_CKPT) or bool(MORL_CKPT)
if _need_torch:
    try:
        import torch  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])

# Sanity report
print('REPO_DIR    =', REPO_DIR)
print('PCT_CKPT    =', PCT_CKPT, '(exists)' if PCT_CKPT and pathlib.Path(PCT_CKPT).exists() else '(SKIP)')
print('MORL_CKPT   =', MORL_CKPT, '(exists)' if MORL_CKPT and pathlib.Path(MORL_CKPT).exists() else '(SKIP)')
print('SEEDS       =', SEEDS)
print('SUITES      =', SUITES_TO_RUN)
print('CPSAT_TIME  =', CPSAT_TIME, 's   workers =', CPSAT_WORKERS)
print('RUN_GA      =', RUN_GA)

In [ ]:
import random
from app.catalog.loader import get_container, get_cargo_preset
from app.data.alexandria_sampler import AlexandriaSampler, SamplerConfig

# Suite specs — kept minimal; tweak counts here if a sweep is too slow.
SUITE_SPECS = {
    'small_easy':   dict(items=12, container='40HC', kind='presets',  preset='eur_pallet_light'),
    'medium_mixed': dict(items=30, container='40HC', kind='sampler',  strategy='mixed'),
    'large_mixed':  dict(items=60, container='40HC', kind='sampler',  strategy='mixed'),
    '20gp_tight':   dict(items=20, container='20GP', kind='sampler',  strategy='mixed'),
    'hazmat_heavy': dict(items=24, container='40HC', kind='hazmat'),
    'reefer':       dict(items=12, container='20RF', kind='presets',  preset='reefer_fruit_pallet'),
}

HAZMAT_MIX = ['steel_drum_200l', 'hazmat_corrosive_drum', 'carton_small', 'eur_pallet_light']


def build_voyage(suite_name, seed):
    cfg = SUITE_SPECS[suite_name]
    cont = get_container(cfg['container'])
    if cfg['kind'] == 'presets':
        items = [
            get_cargo_preset(cfg['preset'], item_id=f'{suite_name[:3]}-{seed}-{i:03d}')
            for i in range(cfg['items'])
        ]
    elif cfg['kind'] == 'hazmat':
        rng = random.Random(seed)
        items = [
            get_cargo_preset(rng.choice(HAZMAT_MIX), item_id=f'haz-{seed}-{i:03d}')
            for i in range(cfg['items'])
        ]
    else:  # sampler
        s = AlexandriaSampler(SamplerConfig(
            n_items=cfg['items'], strategy=cfg['strategy'], seed=seed,
        ))
        items = s.sample()
    return cont, items


# Quick sanity print
for name in SUITES_TO_RUN:
    cont, items = build_voyage(name, SEEDS[0])
    print(f'{name:<14} container={cont.code.value}  items={len(items)}  '
          f'first_id={items[0].id if items else "—"}')

In [ ]:
import time, csv, statistics
from app.algorithms import get_algorithm
from app.algorithms.base import solve


def _ae(placements, container):
    if not placements:
        return 0.0
    door = 0.20 * container.internal.length_mm
    return sum(
        1 for p in placements
        if (p.position.x_mm + p.rotated_dimensions.length_mm / 2) <= door
    ) / len(placements)


def _algo_specs():
    specs = [
        ('bl', {}),
        ('extreme_points', {}),
        ('baf', {}),
        ('bssf', {}),
        ('blsf', {}),
        ('cpsat', {'time_limit_s': CPSAT_TIME, 'num_search_workers': CPSAT_WORKERS, 'enforce_imdg': True}),
    ]
    if RUN_GA:
        specs.insert(5, ('ga', {}))
    if PCT_CKPT and pathlib.Path(PCT_CKPT).exists():
        specs.append(('pct', {'weights_path': PCT_CKPT}))
    if MORL_CKPT and pathlib.Path(MORL_CKPT).exists():
        specs.append(('morl_pct', {'weights_path': MORL_CKPT, 'preference': MORL_PREFERENCE}))
    return specs


def run_one(code, kwargs, cont, items):
    algo = get_algorithm(code, **kwargs)
    t0 = time.perf_counter()
    if hasattr(algo, 'prepare'):
        algo.prepare(cont, items)
    res, _ = solve(algorithm=algo, container=cont, items=items)
    elapsed = time.perf_counter() - t0
    k = res.kpis
    return {
        'util_pct': 100 * k.utilization,
        'placed_pct': 100 * len(res.placements) / max(len(items), 1),
        'access_eff': _ae(res.placements, cont),
        'stability_score': max(0.0, (len(res.placements) - k.unstable_count) / max(len(res.placements), 1)),
        'cog_long_abs': abs(k.cog_long_dev),
        'weight_pct': 100 * k.weight_used,
        'elapsed_s': elapsed,
        'cpsat_status': str(algo.meta.get('cpsat_status', '')) if code == 'cpsat' else '',
    }


algo_specs = _algo_specs()
print('Algorithms:', [c for c, _ in algo_specs])
print('Total runs:', len(SUITES_TO_RUN) * len(SEEDS) * len(algo_specs))

eval_rows = []
t_global = time.perf_counter()
for suite_name in SUITES_TO_RUN:
    spec = SUITE_SPECS[suite_name]
    print(f'\n========== {suite_name}  ({spec["container"]}, {spec["items"]} items) ==========')
    for seed in SEEDS:
        cont, items = build_voyage(suite_name, seed)
        print(f'-- seed {seed} --')
        for code, kwargs in algo_specs:
            try:
                m = run_one(code, kwargs, cont, items)
            except Exception as e:
                print(f'  {code:<16} FAILED: {type(e).__name__}: {e}')
                continue
            row = {'suite': suite_name, 'seed': seed, 'algorithm': code, **m}
            eval_rows.append(row)
            tag = f' [{m["cpsat_status"]}]' if m['cpsat_status'] else ''
            print(
                f'  {code:<16} util {m["util_pct"]:>6.2f}%  placed {m["placed_pct"]:>6.2f}%  '
                f'AE {m["access_eff"]:>4.2f}  SS {m["stability_score"]:>4.2f}  '
                f't {m["elapsed_s"]:>7.2f}s{tag}'
            )

print(f'\nTotal eval wall time: {(time.perf_counter() - t_global)/60:.1f} min')

# Save master CSV
out_dir = pathlib.Path(REPO_DIR, 'benchmarks/out')
out_dir.mkdir(parents=True, exist_ok=True)
master_csv = out_dir / f"full_eval_{time.strftime('%Y%m%d_%H%M%S')}.csv"
with master_csv.open('w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(eval_rows[0].keys()))
    w.writeheader()
    w.writerows(eval_rows)
print(f'→ wrote {master_csv}')

### 8a. Per-suite breakdown

For each suite, show the mean of each metric across seeds, sorted by utilisation. The winner column flags the best algorithm per metric.

In [ ]:
import pandas as pd

df = pd.DataFrame(eval_rows)
print('Master CSV had', len(df), 'rows. Suites:', sorted(df['suite'].unique()))

for suite in SUITES_TO_RUN:
    sub = df[df.suite == suite]
    if sub.empty:
        continue
    agg = sub.groupby('algorithm').agg(
        util_mean=('util_pct', 'mean'),
        util_std=('util_pct', 'std'),
        placed_mean=('placed_pct', 'mean'),
        access_eff=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
        time_s=('elapsed_s', 'mean'),
    ).round(2).sort_values('util_mean', ascending=False)
    print(f'\n────── {suite} ──────')
    print(agg.to_string())
    winners = {
        'util':       agg['util_mean'].idxmax(),
        'placed':     agg['placed_mean'].idxmax(),
        'access':     agg['access_eff'].idxmax(),
        'stability':  agg['stability'].idxmax(),
        '|cog|':      agg['cog'].idxmin(),
        'fastest':    agg['time_s'].idxmin(),
    }
    print('  winners:', ' '.join(f'{k}={v}' for k, v in winners.items()))

### 8b. Overall ranking + head-to-head vs CP-SAT

Across all suites/seeds: mean metrics + how often each algorithm beats CP-SAT on utilisation.

In [ ]:
# Overall mean across every (suite, seed)
overall = df.groupby('algorithm').agg(
    util_mean=('util_pct', 'mean'),
    util_std=('util_pct', 'std'),
    placed_mean=('placed_pct', 'mean'),
    access=('access_eff', 'mean'),
    stability=('stability_score', 'mean'),
    cog=('cog_long_abs', 'mean'),
    time_s=('elapsed_s', 'mean'),
).round(2).sort_values('util_mean', ascending=False)
print('=== OVERALL (all suites × seeds) ===')
print(overall.to_string())

# Head-to-head vs CP-SAT on utilisation, per (suite, seed)
print('\n=== HEAD-TO-HEAD vs CP-SAT (wins / ties / losses on util%) ===')
pivot = df.pivot_table(index=['suite', 'seed'], columns='algorithm', values='util_pct')
if 'cpsat' in pivot.columns:
    cps = pivot['cpsat']
    summary = {}
    for col in pivot.columns:
        if col == 'cpsat':
            continue
        diff = pivot[col] - cps
        summary[col] = dict(
            wins=int((diff > 0.5).sum()),
            ties=int(diff.abs().le(0.5).sum()),
            losses=int((diff < -0.5).sum()),
            mean_gap=round(diff.mean(), 2),
        )
    h2h = pd.DataFrame(summary).T[['wins', 'ties', 'losses', 'mean_gap']]
    print(h2h.to_string())
    print('\n(0.5pp tolerance for ties. mean_gap = algorithm_util − cpsat_util, '
          'so a negative value means CP-SAT typically wins by that much.)')
else:
    print('No CP-SAT rows in df — cannot compute head-to-head.')

## 9. MORL-PCT preference sweep (optional)

If `MORL_CKPT` is set, sweep several preference vectors on the `medium_mixed` suite to inspect the Pareto trade-off the conditioned policy is producing. Each row is one preference; columns are the realised metrics.

In [ ]:
# Preferences ordered: [utilisation, access, stability, cog, weight].
# Each row sums to ~1; the conditioned policy should bend metrics in the direction of the dial.
PREF_SWEEP = [
    ('util_max',      [0.80, 0.05, 0.05, 0.05, 0.05]),
    ('balanced',      [0.40, 0.20, 0.20, 0.10, 0.10]),
    ('access_lean',   [0.30, 0.50, 0.10, 0.05, 0.05]),
    ('stability_lean',[0.30, 0.10, 0.50, 0.05, 0.05]),
    ('cog_lean',      [0.30, 0.10, 0.10, 0.45, 0.05]),
]
SWEEP_SUITE = 'medium_mixed'
SWEEP_SEEDS = SEEDS[:2]  # 2 seeds keeps it quick

if not (MORL_CKPT and pathlib.Path(MORL_CKPT).exists()):
    print('Skipped: MORL_CKPT not set or file not found.')
else:
    sweep_rows = []
    for label, pref in PREF_SWEEP:
        for seed in SWEEP_SEEDS:
            cont, items = build_voyage(SWEEP_SUITE, seed)
            m = run_one('morl_pct', {'weights_path': MORL_CKPT, 'preference': pref}, cont, items)
            sweep_rows.append({'preference': label, 'seed': seed, **m})
            print(f'  {label:<16} seed={seed}  util={m["util_pct"]:.2f}  '
                  f'AE={m["access_eff"]:.2f}  SS={m["stability_score"]:.2f}  '
                  f'|cog|={m["cog_long_abs"]:.3f}  t={m["elapsed_s"]:.2f}s')

    sweep_df = pd.DataFrame(sweep_rows)
    print('\n=== MORL preference sweep — mean over seeds ===')
    sweep_agg = sweep_df.groupby('preference').agg(
        util=('util_pct', 'mean'),
        placed=('placed_pct', 'mean'),
        access=('access_eff', 'mean'),
        stability=('stability_score', 'mean'),
        cog=('cog_long_abs', 'mean'),
    ).round(3)
    # Re-order rows to match PREF_SWEEP definition order
    sweep_agg = sweep_agg.reindex([lbl for lbl, _ in PREF_SWEEP])
    print(sweep_agg.to_string())
    print('\nInterpretation: each row dials one objective. If MORL truly conditions on '
          'preference, util_max should top util, access_lean should top access, etc. '
          'If every row looks identical, the FiLM gating isn\'t reading the preference.')
    sweep_csv = out_dir / f"morl_pref_sweep_{time.strftime('%Y%m%d_%H%M%S')}.csv"
    sweep_df.to_csv(sweep_csv, index=False)
    print(f'→ wrote {sweep_csv}')